# Dataset Assembly

### Purpose

Assemble the final modeling datasets by aligning the processed modality features and outcome labels according to the predefined train, validation, and test splits.

### Objectives

- Load split definitions and processed modality feature matrices.
- Align clinical and RNA features by sample ID within each split.
- Construct target vectors (y) using the same sample ordering.
- Validate dataset integrity (sample counts, ordering, feature dimensions).
- Produce the final model-ready inputs for downstream training and evaluation.


### Workflow

1. Load split-specific processed modality matrices  
   - Load the already-preprocessed train, validation, and test feature matrices for each modality.  
   - Examples:  
       • `X_clin_train_df`, `X_clin_val_df`, `X_clin_test_df`  
       • `X_rna_train_df`, `X_rna_val_df`, `X_rna_test_df`

2. Align samples by ID within each split  
   - Confirm that clinical and RNA matrices have identical sample IDs in identical order.  
   - Reindex only if needed to enforce exact alignment.  
   - Expected condition:  
       • `X_clin_train_df.index == X_rna_train_df.index`  
       • `X_clin_val_df.index == X_rna_val_df.index`  
       • `X_clin_test_df.index == X_rna_test_df.index`

3. Build target vectors  
   - Load survival or outcome labels and construct targets for each split.  
   - Targets must use the same sample IDs and row order as the feature matrices.  
   - Examples:  
       • `y_train`  
       • `y_val`  
       • `y_test`

4. Validate dataset invariants  
   - Perform sanity checks to confirm that assembled datasets are consistent.  
   - Checks should include:  
       • identical sample counts across modalities and targets  
       • identical sample ordering  
       • no duplicate sample IDs  
       • no missing values introduced during alignment  
       • expected feature counts for each modality

5. Save final model inputs  
   - Write the assembled datasets for downstream modeling.  
   - Save modality-specific inputs for each split.  
   - Optionally also save concatenated feature matrices for baseline models.

6. Test assemble dataset module

In [49]:
from pathlib import Path
import pandas as pd
import json
import subprocess
import sys

## 1. Load split-specific processed modality matrices  
   - Load the already-preprocessed train, validation, and test feature matrices for each modality.  
   - Examples:  
       • `X_clin_train_df`, `X_clin_val_df`, `X_clin_test_df`  
       • `X_rna_train_df`, `X_rna_val_df`, `X_rna_test_df`

In [50]:
# Load the split-specific processed matrices produced by the preprocessing scripts.

clinical_dir = Path("../data/processed/clinical")
rna_dir = Path("../data/processed/rna")

X_clin_train_df = pd.read_parquet(clinical_dir / "train" / "X_clinical.parquet")
X_clin_val_df = pd.read_parquet(clinical_dir / "val" / "X_clinical.parquet")
X_clin_test_df = pd.read_parquet(clinical_dir / "test" / "X_clinical.parquet")

X_rna_train_df = pd.read_parquet(rna_dir / "train" / "X_rna.parquet")
X_rna_val_df = pd.read_parquet(rna_dir / "val" / "X_rna.parquet")
X_rna_test_df = pd.read_parquet(rna_dir / "test" / "X_rna.parquet")

# Basic sanity checks: each split should contain rows and be indexed by sample ID.
assert X_clin_train_df.shape[0] > 0 and X_rna_train_df.shape[0] > 0
assert X_clin_train_df.index.name == "sample" and X_rna_train_df.index.name == "sample"

display(X_clin_train_df.head(), X_clin_train_df.tail())
display(X_rna_train_df.head(), X_rna_train_df.tail())

print(
    f"clinical: train={X_clin_train_df.shape}, val={X_clin_val_df.shape}, test={X_clin_test_df.shape}"
)
print(
    f"rna: train={X_rna_train_df.shape}, val={X_rna_val_df.shape}, test={X_rna_test_df.shape}"
)

,year_of_birth.demographic,year_of_diagnosis.diagnoses,age_at_earliest_diagnosis_in_years.diagnoses.xena_derived,days_to_collection.samples,initial_weight.samples,oct_embedded.samples,ajcc_pathologic_m.diagnoses_M0,ajcc_pathologic_m.diagnoses_M1,ajcc_pathologic_m.diagnoses_MX,ajcc_pathologic_m.diagnoses_cM0 (i+),...,prior_treatment.diagnoses_Yes,race.demographic_asian,race.demographic_black or african american,race.demographic_not reported,race.demographic_white,"tissue_or_organ_of_origin.diagnoses_Breast, NOS",tissue_or_organ_of_origin.diagnoses_Lower-inner quadrant of breast,tissue_or_organ_of_origin.diagnoses_Overlapping lesion of breast,"treatment_type.treatments.diagnoses_['Pharmaceutical Therapy, NOS', 'Radiation Therapy, NOS']","treatment_type.treatments.diagnoses_['Radiation Therapy, NOS', 'Pharmaceutical Therapy, NOS']"
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-AR-A1AN-01A,1960.0,2006.0,46.504110,1463.0,70.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
TCGA-B6-A2IU-01A,1936.0,1998.0,62.753425,4665.0,140.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
TCGA-C8-A12Q-01A,1932.0,2010.0,78.720548,57.0,810.0,1,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
TCGA-BH-A0E9-01B,1953.0,2006.0,53.191781,1357.0,480.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
TCGA-A2-A0ST-01A,1941.0,2003.0,62.800000,2597.0,210.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0


,year_of_birth.demographic,year_of_diagnosis.diagnoses,age_at_earliest_diagnosis_in_years.diagnoses.xena_derived,days_to_collection.samples,initial_weight.samples,oct_embedded.samples,ajcc_pathologic_m.diagnoses_M0,ajcc_pathologic_m.diagnoses_M1,ajcc_pathologic_m.diagnoses_MX,ajcc_pathologic_m.diagnoses_cM0 (i+),...,prior_treatment.diagnoses_Yes,race.demographic_asian,race.demographic_black or african american,race.demographic_not reported,race.demographic_white,"tissue_or_organ_of_origin.diagnoses_Breast, NOS",tissue_or_organ_of_origin.diagnoses_Lower-inner quadrant of breast,tissue_or_organ_of_origin.diagnoses_Overlapping lesion of breast,"treatment_type.treatments.diagnoses_['Pharmaceutical Therapy, NOS', 'Radiation Therapy, NOS']","treatment_type.treatments.diagnoses_['Radiation Therapy, NOS', 'Pharmaceutical Therapy, NOS']"
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-GM-A2DM-01A,1947.0,2004.0,57.972603,2464.0,120.0,0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
TCGA-AR-A2LE-01A,1932.0,2001.0,69.860274,3755.0,320.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
TCGA-E2-A1LK-01A,1924.0,2008.0,84.265753,1051.0,70.0,0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
TCGA-E9-A1RB-01A,1971.0,2011.0,40.424658,39.0,490.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
TCGA-B6-A0I6-01A,1941.0,1990.0,49.353425,7362.0,200.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,ENSG00000001167.14,...,ENSG00000288573.1,ENSG00000288585.1,ENSG00000288586.1,ENSG00000288596.2,ENSG00000288600.1,ENSG00000288605.1,ENSG00000288612.1,ENSG00000288658.1,ENSG00000288670.1,ENSG00000288675.1
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-AR-A1AN-01A,-0.921931,0.378658,-1.174161,-0.611187,-0.610914,-0.249239,0.837047,0.323434,0.456512,-0.088932,...,-1.353641,-0.352161,0.247873,-1.847424,-0.651507,-0.606533,-0.440411,0.176202,0.426529,1.225862
TCGA-B6-A2IU-01A,-0.972194,0.665513,-1.350524,2.081968,0.961827,-0.919410,-0.874112,-1.378494,0.926571,0.534901,...,0.012429,1.103978,1.517371,0.609031,0.524253,-0.474512,-0.202529,-0.660575,0.394027,-0.961233
TCGA-C8-A12Q-01A,0.091865,0.330616,0.620092,0.676719,0.367927,0.586679,0.679828,-0.486117,0.479864,-0.354779,...,-1.509343,-0.020435,-0.527875,-0.585608,0.204504,-0.378965,2.682691,-0.551112,0.489490,-1.008443
TCGA-BH-A0E9-01B,1.720857,0.423156,-0.174513,1.528813,1.259411,0.227238,1.043159,0.209277,0.017179,0.802620,...,0.585732,2.531337,-0.092806,0.688908,-0.607627,-0.034685,1.212378,-0.406525,0.965263,0.145407
TCGA-A2-A0ST-01A,0.243425,-0.114109,-0.919132,-1.255508,-0.481986,1.644299,-0.028138,-0.191235,-1.162009,-0.118621,...,0.178092,-0.914768,-1.051409,-0.765611,2.093778,-0.025683,-0.928029,0.542307,-1.556974,-0.272568


,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,ENSG00000001167.14,...,ENSG00000288573.1,ENSG00000288585.1,ENSG00000288586.1,ENSG00000288596.2,ENSG00000288600.1,ENSG00000288605.1,ENSG00000288612.1,ENSG00000288658.1,ENSG00000288670.1,ENSG00000288675.1
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-GM-A2DM-01A,1.360126,-0.908692,-0.565739,-0.864921,-1.522170,-0.233247,-1.520043,1.149233,2.240680,-0.250259,...,0.265048,-0.884606,0.141415,-0.480243,-0.579166,-0.528830,-0.286348,-0.517218,0.045952,0.367215
TCGA-AR-A2LE-01A,1.601834,-0.590718,-1.719977,0.373477,-0.739503,-1.368053,-0.252641,-0.792960,0.841976,-0.580795,...,0.236000,-0.231015,0.883146,-0.100171,-0.352117,-0.606533,-0.897766,3.383043,0.066095,-0.709205
TCGA-E2-A1LK-01A,0.463213,-0.980631,0.258041,-0.955474,-0.335838,-0.187937,-2.414184,0.405145,-0.962193,-0.774558,...,0.110951,-0.607541,0.610262,1.347791,-0.521076,1.184644,1.959450,1.703494,0.349865,-0.400755
TCGA-E9-A1RB-01A,-0.485210,-0.816918,0.765878,0.101420,1.483606,-0.516478,0.009252,-0.096645,-1.336679,-0.984773,...,-1.007468,-0.571601,-1.173723,0.221939,-0.343172,-0.560550,-0.506072,-0.084724,0.648266,-0.360840
TCGA-B6-A0I6-01A,0.494079,-0.980631,0.463717,-1.389508,-0.531046,-0.568328,-0.144039,0.541407,-0.097906,-1.219733,...,0.366692,-0.945813,2.813252,0.383612,-0.208193,0.491286,-1.132417,1.301828,-0.865487,-1.733192


clinical: train=(203, 98), val=(43, 98), test=(44, 98)
rna: train=(203, 25431), val=(43, 25431), test=(44, 25431)


In [51]:
def load_processed_modality_matrices(
    clinical_dir: str | Path,
    rna_dir: str | Path,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load split-specific processed clinical and RNA feature matrices.

    Parameters
    ----------
    clinical_dir : str | Path
        Base directory containing split-specific clinical parquet files.
    rna_dir : str | Path
        Base directory containing split-specific RNA parquet files.

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]
        `X_clin_train_df`, `X_clin_val_df`, `X_clin_test_df`,
        `X_rna_train_df`, `X_rna_val_df`, `X_rna_test_df`.
    """
    clinical_dir = Path(clinical_dir)
    rna_dir = Path(rna_dir)

    # Load the split-specific processed matrices produced by the preprocessing scripts.
    X_clin_train_df = pd.read_parquet(clinical_dir / "train" / "X_clinical.parquet")
    X_clin_val_df = pd.read_parquet(clinical_dir / "val" / "X_clinical.parquet")
    X_clin_test_df = pd.read_parquet(clinical_dir / "test" / "X_clinical.parquet")

    X_rna_train_df = pd.read_parquet(rna_dir / "train" / "X_rna.parquet")
    X_rna_val_df = pd.read_parquet(rna_dir / "val" / "X_rna.parquet")
    X_rna_test_df = pd.read_parquet(rna_dir / "test" / "X_rna.parquet")

    # Basic sanity checks: each split should contain rows and be indexed by sample ID.
    assert X_clin_train_df.shape[0] > 0 and X_rna_train_df.shape[0] > 0
    assert X_clin_train_df.index.name == "sample" and X_rna_train_df.index.name == "sample"

    display(X_clin_train_df.head(), X_clin_train_df.tail())
    display(X_rna_train_df.head(), X_rna_train_df.tail())

    print(
        f"clinical: train={X_clin_train_df.shape}, val={X_clin_val_df.shape}, test={X_clin_test_df.shape}"
    )
    print(
        f"rna: train={X_rna_train_df.shape}, val={X_rna_val_df.shape}, test={X_rna_test_df.shape}"
    )

    return (
        X_clin_train_df,
        X_clin_val_df,
        X_clin_test_df,
        X_rna_train_df,
        X_rna_val_df,
        X_rna_test_df,
    )


(
    X_clin_train_loaded_df,
    X_clin_val_loaded_df,
    X_clin_test_loaded_df,
    X_rna_train_loaded_df,
    X_rna_val_loaded_df,
    X_rna_test_loaded_df,
) = load_processed_modality_matrices(clinical_dir=clinical_dir, rna_dir=rna_dir)

assert X_clin_train_loaded_df.equals(X_clin_train_df)
assert X_clin_val_loaded_df.equals(X_clin_val_df)
assert X_clin_test_loaded_df.equals(X_clin_test_df)

assert X_rna_train_loaded_df.equals(X_rna_train_df)
assert X_rna_val_loaded_df.equals(X_rna_val_df)
assert X_rna_test_loaded_df.equals(X_rna_test_df)

print("Refactored function output matches the existing notebook objects.")


,year_of_birth.demographic,year_of_diagnosis.diagnoses,age_at_earliest_diagnosis_in_years.diagnoses.xena_derived,days_to_collection.samples,initial_weight.samples,oct_embedded.samples,ajcc_pathologic_m.diagnoses_M0,ajcc_pathologic_m.diagnoses_M1,ajcc_pathologic_m.diagnoses_MX,ajcc_pathologic_m.diagnoses_cM0 (i+),...,prior_treatment.diagnoses_Yes,race.demographic_asian,race.demographic_black or african american,race.demographic_not reported,race.demographic_white,"tissue_or_organ_of_origin.diagnoses_Breast, NOS",tissue_or_organ_of_origin.diagnoses_Lower-inner quadrant of breast,tissue_or_organ_of_origin.diagnoses_Overlapping lesion of breast,"treatment_type.treatments.diagnoses_['Pharmaceutical Therapy, NOS', 'Radiation Therapy, NOS']","treatment_type.treatments.diagnoses_['Radiation Therapy, NOS', 'Pharmaceutical Therapy, NOS']"
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-AR-A1AN-01A,1960.0,2006.0,46.504110,1463.0,70.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
TCGA-B6-A2IU-01A,1936.0,1998.0,62.753425,4665.0,140.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
TCGA-C8-A12Q-01A,1932.0,2010.0,78.720548,57.0,810.0,1,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
TCGA-BH-A0E9-01B,1953.0,2006.0,53.191781,1357.0,480.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
TCGA-A2-A0ST-01A,1941.0,2003.0,62.800000,2597.0,210.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0


,year_of_birth.demographic,year_of_diagnosis.diagnoses,age_at_earliest_diagnosis_in_years.diagnoses.xena_derived,days_to_collection.samples,initial_weight.samples,oct_embedded.samples,ajcc_pathologic_m.diagnoses_M0,ajcc_pathologic_m.diagnoses_M1,ajcc_pathologic_m.diagnoses_MX,ajcc_pathologic_m.diagnoses_cM0 (i+),...,prior_treatment.diagnoses_Yes,race.demographic_asian,race.demographic_black or african american,race.demographic_not reported,race.demographic_white,"tissue_or_organ_of_origin.diagnoses_Breast, NOS",tissue_or_organ_of_origin.diagnoses_Lower-inner quadrant of breast,tissue_or_organ_of_origin.diagnoses_Overlapping lesion of breast,"treatment_type.treatments.diagnoses_['Pharmaceutical Therapy, NOS', 'Radiation Therapy, NOS']","treatment_type.treatments.diagnoses_['Radiation Therapy, NOS', 'Pharmaceutical Therapy, NOS']"
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-GM-A2DM-01A,1947.0,2004.0,57.972603,2464.0,120.0,0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
TCGA-AR-A2LE-01A,1932.0,2001.0,69.860274,3755.0,320.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
TCGA-E2-A1LK-01A,1924.0,2008.0,84.265753,1051.0,70.0,0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
TCGA-E9-A1RB-01A,1971.0,2011.0,40.424658,39.0,490.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
TCGA-B6-A0I6-01A,1941.0,1990.0,49.353425,7362.0,200.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,ENSG00000001167.14,...,ENSG00000288573.1,ENSG00000288585.1,ENSG00000288586.1,ENSG00000288596.2,ENSG00000288600.1,ENSG00000288605.1,ENSG00000288612.1,ENSG00000288658.1,ENSG00000288670.1,ENSG00000288675.1
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-AR-A1AN-01A,-0.921931,0.378658,-1.174161,-0.611187,-0.610914,-0.249239,0.837047,0.323434,0.456512,-0.088932,...,-1.353641,-0.352161,0.247873,-1.847424,-0.651507,-0.606533,-0.440411,0.176202,0.426529,1.225862
TCGA-B6-A2IU-01A,-0.972194,0.665513,-1.350524,2.081968,0.961827,-0.919410,-0.874112,-1.378494,0.926571,0.534901,...,0.012429,1.103978,1.517371,0.609031,0.524253,-0.474512,-0.202529,-0.660575,0.394027,-0.961233
TCGA-C8-A12Q-01A,0.091865,0.330616,0.620092,0.676719,0.367927,0.586679,0.679828,-0.486117,0.479864,-0.354779,...,-1.509343,-0.020435,-0.527875,-0.585608,0.204504,-0.378965,2.682691,-0.551112,0.489490,-1.008443
TCGA-BH-A0E9-01B,1.720857,0.423156,-0.174513,1.528813,1.259411,0.227238,1.043159,0.209277,0.017179,0.802620,...,0.585732,2.531337,-0.092806,0.688908,-0.607627,-0.034685,1.212378,-0.406525,0.965263,0.145407
TCGA-A2-A0ST-01A,0.243425,-0.114109,-0.919132,-1.255508,-0.481986,1.644299,-0.028138,-0.191235,-1.162009,-0.118621,...,0.178092,-0.914768,-1.051409,-0.765611,2.093778,-0.025683,-0.928029,0.542307,-1.556974,-0.272568


,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,ENSG00000001167.14,...,ENSG00000288573.1,ENSG00000288585.1,ENSG00000288586.1,ENSG00000288596.2,ENSG00000288600.1,ENSG00000288605.1,ENSG00000288612.1,ENSG00000288658.1,ENSG00000288670.1,ENSG00000288675.1
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-GM-A2DM-01A,1.360126,-0.908692,-0.565739,-0.864921,-1.522170,-0.233247,-1.520043,1.149233,2.240680,-0.250259,...,0.265048,-0.884606,0.141415,-0.480243,-0.579166,-0.528830,-0.286348,-0.517218,0.045952,0.367215
TCGA-AR-A2LE-01A,1.601834,-0.590718,-1.719977,0.373477,-0.739503,-1.368053,-0.252641,-0.792960,0.841976,-0.580795,...,0.236000,-0.231015,0.883146,-0.100171,-0.352117,-0.606533,-0.897766,3.383043,0.066095,-0.709205
TCGA-E2-A1LK-01A,0.463213,-0.980631,0.258041,-0.955474,-0.335838,-0.187937,-2.414184,0.405145,-0.962193,-0.774558,...,0.110951,-0.607541,0.610262,1.347791,-0.521076,1.184644,1.959450,1.703494,0.349865,-0.400755
TCGA-E9-A1RB-01A,-0.485210,-0.816918,0.765878,0.101420,1.483606,-0.516478,0.009252,-0.096645,-1.336679,-0.984773,...,-1.007468,-0.571601,-1.173723,0.221939,-0.343172,-0.560550,-0.506072,-0.084724,0.648266,-0.360840
TCGA-B6-A0I6-01A,0.494079,-0.980631,0.463717,-1.389508,-0.531046,-0.568328,-0.144039,0.541407,-0.097906,-1.219733,...,0.366692,-0.945813,2.813252,0.383612,-0.208193,0.491286,-1.132417,1.301828,-0.865487,-1.733192


clinical: train=(203, 98), val=(43, 98), test=(44, 98)
rna: train=(203, 25431), val=(43, 25431), test=(44, 25431)
Refactored function output matches the existing notebook objects.


## 2. Align samples by ID within each split  
   - Confirm that clinical and RNA matrices have identical sample IDs in identical order.  
   - Reindex only if needed to enforce exact alignment.  
   - Expected condition:  
       • `X_clin_train_df.index == X_rna_train_df.index`  
       • `X_clin_val_df.index == X_rna_val_df.index`  
       • `X_clin_test_df.index == X_rna_test_df.index`

In [52]:
# Reindex RNA to clinical within each split only if ordering differs.

if not X_clin_train_df.index.equals(X_rna_train_df.index):
    X_rna_train_df = X_rna_train_df.reindex(X_clin_train_df.index)

if not X_clin_val_df.index.equals(X_rna_val_df.index):
    X_rna_val_df = X_rna_val_df.reindex(X_clin_val_df.index)

if not X_clin_test_df.index.equals(X_rna_test_df.index):
    X_rna_test_df = X_rna_test_df.reindex(X_clin_test_df.index)

# Validate exact sample alignment after any needed reindexing.
assert X_clin_train_df.index.equals(X_rna_train_df.index)
assert X_clin_val_df.index.equals(X_rna_val_df.index)
assert X_clin_test_df.index.equals(X_rna_test_df.index)

display(X_clin_train_df.head(), X_clin_train_df.tail())
display(X_rna_train_df.head(), X_rna_train_df.tail())


,year_of_birth.demographic,year_of_diagnosis.diagnoses,age_at_earliest_diagnosis_in_years.diagnoses.xena_derived,days_to_collection.samples,initial_weight.samples,oct_embedded.samples,ajcc_pathologic_m.diagnoses_M0,ajcc_pathologic_m.diagnoses_M1,ajcc_pathologic_m.diagnoses_MX,ajcc_pathologic_m.diagnoses_cM0 (i+),...,prior_treatment.diagnoses_Yes,race.demographic_asian,race.demographic_black or african american,race.demographic_not reported,race.demographic_white,"tissue_or_organ_of_origin.diagnoses_Breast, NOS",tissue_or_organ_of_origin.diagnoses_Lower-inner quadrant of breast,tissue_or_organ_of_origin.diagnoses_Overlapping lesion of breast,"treatment_type.treatments.diagnoses_['Pharmaceutical Therapy, NOS', 'Radiation Therapy, NOS']","treatment_type.treatments.diagnoses_['Radiation Therapy, NOS', 'Pharmaceutical Therapy, NOS']"
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-AR-A1AN-01A,1960.0,2006.0,46.504110,1463.0,70.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
TCGA-B6-A2IU-01A,1936.0,1998.0,62.753425,4665.0,140.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
TCGA-C8-A12Q-01A,1932.0,2010.0,78.720548,57.0,810.0,1,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
TCGA-BH-A0E9-01B,1953.0,2006.0,53.191781,1357.0,480.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
TCGA-A2-A0ST-01A,1941.0,2003.0,62.800000,2597.0,210.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0


,year_of_birth.demographic,year_of_diagnosis.diagnoses,age_at_earliest_diagnosis_in_years.diagnoses.xena_derived,days_to_collection.samples,initial_weight.samples,oct_embedded.samples,ajcc_pathologic_m.diagnoses_M0,ajcc_pathologic_m.diagnoses_M1,ajcc_pathologic_m.diagnoses_MX,ajcc_pathologic_m.diagnoses_cM0 (i+),...,prior_treatment.diagnoses_Yes,race.demographic_asian,race.demographic_black or african american,race.demographic_not reported,race.demographic_white,"tissue_or_organ_of_origin.diagnoses_Breast, NOS",tissue_or_organ_of_origin.diagnoses_Lower-inner quadrant of breast,tissue_or_organ_of_origin.diagnoses_Overlapping lesion of breast,"treatment_type.treatments.diagnoses_['Pharmaceutical Therapy, NOS', 'Radiation Therapy, NOS']","treatment_type.treatments.diagnoses_['Radiation Therapy, NOS', 'Pharmaceutical Therapy, NOS']"
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-GM-A2DM-01A,1947.0,2004.0,57.972603,2464.0,120.0,0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
TCGA-AR-A2LE-01A,1932.0,2001.0,69.860274,3755.0,320.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
TCGA-E2-A1LK-01A,1924.0,2008.0,84.265753,1051.0,70.0,0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
TCGA-E9-A1RB-01A,1971.0,2011.0,40.424658,39.0,490.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
TCGA-B6-A0I6-01A,1941.0,1990.0,49.353425,7362.0,200.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,ENSG00000001167.14,...,ENSG00000288573.1,ENSG00000288585.1,ENSG00000288586.1,ENSG00000288596.2,ENSG00000288600.1,ENSG00000288605.1,ENSG00000288612.1,ENSG00000288658.1,ENSG00000288670.1,ENSG00000288675.1
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-AR-A1AN-01A,-0.921931,0.378658,-1.174161,-0.611187,-0.610914,-0.249239,0.837047,0.323434,0.456512,-0.088932,...,-1.353641,-0.352161,0.247873,-1.847424,-0.651507,-0.606533,-0.440411,0.176202,0.426529,1.225862
TCGA-B6-A2IU-01A,-0.972194,0.665513,-1.350524,2.081968,0.961827,-0.919410,-0.874112,-1.378494,0.926571,0.534901,...,0.012429,1.103978,1.517371,0.609031,0.524253,-0.474512,-0.202529,-0.660575,0.394027,-0.961233
TCGA-C8-A12Q-01A,0.091865,0.330616,0.620092,0.676719,0.367927,0.586679,0.679828,-0.486117,0.479864,-0.354779,...,-1.509343,-0.020435,-0.527875,-0.585608,0.204504,-0.378965,2.682691,-0.551112,0.489490,-1.008443
TCGA-BH-A0E9-01B,1.720857,0.423156,-0.174513,1.528813,1.259411,0.227238,1.043159,0.209277,0.017179,0.802620,...,0.585732,2.531337,-0.092806,0.688908,-0.607627,-0.034685,1.212378,-0.406525,0.965263,0.145407
TCGA-A2-A0ST-01A,0.243425,-0.114109,-0.919132,-1.255508,-0.481986,1.644299,-0.028138,-0.191235,-1.162009,-0.118621,...,0.178092,-0.914768,-1.051409,-0.765611,2.093778,-0.025683,-0.928029,0.542307,-1.556974,-0.272568


,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,ENSG00000001167.14,...,ENSG00000288573.1,ENSG00000288585.1,ENSG00000288586.1,ENSG00000288596.2,ENSG00000288600.1,ENSG00000288605.1,ENSG00000288612.1,ENSG00000288658.1,ENSG00000288670.1,ENSG00000288675.1
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-GM-A2DM-01A,1.360126,-0.908692,-0.565739,-0.864921,-1.522170,-0.233247,-1.520043,1.149233,2.240680,-0.250259,...,0.265048,-0.884606,0.141415,-0.480243,-0.579166,-0.528830,-0.286348,-0.517218,0.045952,0.367215
TCGA-AR-A2LE-01A,1.601834,-0.590718,-1.719977,0.373477,-0.739503,-1.368053,-0.252641,-0.792960,0.841976,-0.580795,...,0.236000,-0.231015,0.883146,-0.100171,-0.352117,-0.606533,-0.897766,3.383043,0.066095,-0.709205
TCGA-E2-A1LK-01A,0.463213,-0.980631,0.258041,-0.955474,-0.335838,-0.187937,-2.414184,0.405145,-0.962193,-0.774558,...,0.110951,-0.607541,0.610262,1.347791,-0.521076,1.184644,1.959450,1.703494,0.349865,-0.400755
TCGA-E9-A1RB-01A,-0.485210,-0.816918,0.765878,0.101420,1.483606,-0.516478,0.009252,-0.096645,-1.336679,-0.984773,...,-1.007468,-0.571601,-1.173723,0.221939,-0.343172,-0.560550,-0.506072,-0.084724,0.648266,-0.360840
TCGA-B6-A0I6-01A,0.494079,-0.980631,0.463717,-1.389508,-0.531046,-0.568328,-0.144039,0.541407,-0.097906,-1.219733,...,0.366692,-0.945813,2.813252,0.383612,-0.208193,0.491286,-1.132417,1.301828,-0.865487,-1.733192


In [53]:
def align_modalities_within_splits(
    X_clin_train_df: pd.DataFrame,
    X_clin_val_df: pd.DataFrame,
    X_clin_test_df: pd.DataFrame,
    X_rna_train_df: pd.DataFrame,
    X_rna_val_df: pd.DataFrame,
    X_rna_test_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Align RNA sample ordering to clinical sample ordering within each split.

    Parameters
    ----------
    X_clin_train_df, X_clin_val_df, X_clin_test_df : pd.DataFrame
        Clinical feature matrices for the train, validation, and test splits.
    X_rna_train_df, X_rna_val_df, X_rna_test_df : pd.DataFrame
        RNA feature matrices for the train, validation, and test splits.

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]
        `X_clin_train_df`, `X_clin_val_df`, `X_clin_test_df`,
        `X_rna_train_df`, `X_rna_val_df`, `X_rna_test_df`, with RNA reindexed
        to clinical ordering only when needed.
    """
    if not X_clin_train_df.index.equals(X_rna_train_df.index):
        X_rna_train_df = X_rna_train_df.reindex(X_clin_train_df.index)

    if not X_clin_val_df.index.equals(X_rna_val_df.index):
        X_rna_val_df = X_rna_val_df.reindex(X_clin_val_df.index)

    if not X_clin_test_df.index.equals(X_rna_test_df.index):
        X_rna_test_df = X_rna_test_df.reindex(X_clin_test_df.index)

    # Validate exact sample alignment after any needed reindexing.
    assert X_clin_train_df.index.equals(X_rna_train_df.index)
    assert X_clin_val_df.index.equals(X_rna_val_df.index)
    assert X_clin_test_df.index.equals(X_rna_test_df.index)

    display(X_clin_train_df.head(), X_clin_train_df.tail())
    display(X_rna_train_df.head(), X_rna_train_df.tail())

    return (
        X_clin_train_df,
        X_clin_val_df,
        X_clin_test_df,
        X_rna_train_df,
        X_rna_val_df,
        X_rna_test_df,
    )


(
    X_clin_train_aligned_df,
    X_clin_val_aligned_df,
    X_clin_test_aligned_df,
    X_rna_train_aligned_df,
    X_rna_val_aligned_df,
    X_rna_test_aligned_df,
) = align_modalities_within_splits(
    X_clin_train_df=X_clin_train_df,
    X_clin_val_df=X_clin_val_df,
    X_clin_test_df=X_clin_test_df,
    X_rna_train_df=X_rna_train_df,
    X_rna_val_df=X_rna_val_df,
    X_rna_test_df=X_rna_test_df,
)

assert X_clin_train_aligned_df.equals(X_clin_train_df)
assert X_clin_val_aligned_df.equals(X_clin_val_df)
assert X_clin_test_aligned_df.equals(X_clin_test_df)

assert X_rna_train_aligned_df.equals(X_rna_train_df)
assert X_rna_val_aligned_df.equals(X_rna_val_df)
assert X_rna_test_aligned_df.equals(X_rna_test_df)

print("Refactored function output matches the existing notebook objects.")


,year_of_birth.demographic,year_of_diagnosis.diagnoses,age_at_earliest_diagnosis_in_years.diagnoses.xena_derived,days_to_collection.samples,initial_weight.samples,oct_embedded.samples,ajcc_pathologic_m.diagnoses_M0,ajcc_pathologic_m.diagnoses_M1,ajcc_pathologic_m.diagnoses_MX,ajcc_pathologic_m.diagnoses_cM0 (i+),...,prior_treatment.diagnoses_Yes,race.demographic_asian,race.demographic_black or african american,race.demographic_not reported,race.demographic_white,"tissue_or_organ_of_origin.diagnoses_Breast, NOS",tissue_or_organ_of_origin.diagnoses_Lower-inner quadrant of breast,tissue_or_organ_of_origin.diagnoses_Overlapping lesion of breast,"treatment_type.treatments.diagnoses_['Pharmaceutical Therapy, NOS', 'Radiation Therapy, NOS']","treatment_type.treatments.diagnoses_['Radiation Therapy, NOS', 'Pharmaceutical Therapy, NOS']"
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-AR-A1AN-01A,1960.0,2006.0,46.504110,1463.0,70.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
TCGA-B6-A2IU-01A,1936.0,1998.0,62.753425,4665.0,140.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
TCGA-C8-A12Q-01A,1932.0,2010.0,78.720548,57.0,810.0,1,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
TCGA-BH-A0E9-01B,1953.0,2006.0,53.191781,1357.0,480.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
TCGA-A2-A0ST-01A,1941.0,2003.0,62.800000,2597.0,210.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0


,year_of_birth.demographic,year_of_diagnosis.diagnoses,age_at_earliest_diagnosis_in_years.diagnoses.xena_derived,days_to_collection.samples,initial_weight.samples,oct_embedded.samples,ajcc_pathologic_m.diagnoses_M0,ajcc_pathologic_m.diagnoses_M1,ajcc_pathologic_m.diagnoses_MX,ajcc_pathologic_m.diagnoses_cM0 (i+),...,prior_treatment.diagnoses_Yes,race.demographic_asian,race.demographic_black or african american,race.demographic_not reported,race.demographic_white,"tissue_or_organ_of_origin.diagnoses_Breast, NOS",tissue_or_organ_of_origin.diagnoses_Lower-inner quadrant of breast,tissue_or_organ_of_origin.diagnoses_Overlapping lesion of breast,"treatment_type.treatments.diagnoses_['Pharmaceutical Therapy, NOS', 'Radiation Therapy, NOS']","treatment_type.treatments.diagnoses_['Radiation Therapy, NOS', 'Pharmaceutical Therapy, NOS']"
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-GM-A2DM-01A,1947.0,2004.0,57.972603,2464.0,120.0,0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
TCGA-AR-A2LE-01A,1932.0,2001.0,69.860274,3755.0,320.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
TCGA-E2-A1LK-01A,1924.0,2008.0,84.265753,1051.0,70.0,0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
TCGA-E9-A1RB-01A,1971.0,2011.0,40.424658,39.0,490.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
TCGA-B6-A0I6-01A,1941.0,1990.0,49.353425,7362.0,200.0,1,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,ENSG00000001167.14,...,ENSG00000288573.1,ENSG00000288585.1,ENSG00000288586.1,ENSG00000288596.2,ENSG00000288600.1,ENSG00000288605.1,ENSG00000288612.1,ENSG00000288658.1,ENSG00000288670.1,ENSG00000288675.1
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-AR-A1AN-01A,-0.921931,0.378658,-1.174161,-0.611187,-0.610914,-0.249239,0.837047,0.323434,0.456512,-0.088932,...,-1.353641,-0.352161,0.247873,-1.847424,-0.651507,-0.606533,-0.440411,0.176202,0.426529,1.225862
TCGA-B6-A2IU-01A,-0.972194,0.665513,-1.350524,2.081968,0.961827,-0.919410,-0.874112,-1.378494,0.926571,0.534901,...,0.012429,1.103978,1.517371,0.609031,0.524253,-0.474512,-0.202529,-0.660575,0.394027,-0.961233
TCGA-C8-A12Q-01A,0.091865,0.330616,0.620092,0.676719,0.367927,0.586679,0.679828,-0.486117,0.479864,-0.354779,...,-1.509343,-0.020435,-0.527875,-0.585608,0.204504,-0.378965,2.682691,-0.551112,0.489490,-1.008443
TCGA-BH-A0E9-01B,1.720857,0.423156,-0.174513,1.528813,1.259411,0.227238,1.043159,0.209277,0.017179,0.802620,...,0.585732,2.531337,-0.092806,0.688908,-0.607627,-0.034685,1.212378,-0.406525,0.965263,0.145407
TCGA-A2-A0ST-01A,0.243425,-0.114109,-0.919132,-1.255508,-0.481986,1.644299,-0.028138,-0.191235,-1.162009,-0.118621,...,0.178092,-0.914768,-1.051409,-0.765611,2.093778,-0.025683,-0.928029,0.542307,-1.556974,-0.272568


,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,ENSG00000001167.14,...,ENSG00000288573.1,ENSG00000288585.1,ENSG00000288586.1,ENSG00000288596.2,ENSG00000288600.1,ENSG00000288605.1,ENSG00000288612.1,ENSG00000288658.1,ENSG00000288670.1,ENSG00000288675.1
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-GM-A2DM-01A,1.360126,-0.908692,-0.565739,-0.864921,-1.522170,-0.233247,-1.520043,1.149233,2.240680,-0.250259,...,0.265048,-0.884606,0.141415,-0.480243,-0.579166,-0.528830,-0.286348,-0.517218,0.045952,0.367215
TCGA-AR-A2LE-01A,1.601834,-0.590718,-1.719977,0.373477,-0.739503,-1.368053,-0.252641,-0.792960,0.841976,-0.580795,...,0.236000,-0.231015,0.883146,-0.100171,-0.352117,-0.606533,-0.897766,3.383043,0.066095,-0.709205
TCGA-E2-A1LK-01A,0.463213,-0.980631,0.258041,-0.955474,-0.335838,-0.187937,-2.414184,0.405145,-0.962193,-0.774558,...,0.110951,-0.607541,0.610262,1.347791,-0.521076,1.184644,1.959450,1.703494,0.349865,-0.400755
TCGA-E9-A1RB-01A,-0.485210,-0.816918,0.765878,0.101420,1.483606,-0.516478,0.009252,-0.096645,-1.336679,-0.984773,...,-1.007468,-0.571601,-1.173723,0.221939,-0.343172,-0.560550,-0.506072,-0.084724,0.648266,-0.360840
TCGA-B6-A0I6-01A,0.494079,-0.980631,0.463717,-1.389508,-0.531046,-0.568328,-0.144039,0.541407,-0.097906,-1.219733,...,0.366692,-0.945813,2.813252,0.383612,-0.208193,0.491286,-1.132417,1.301828,-0.865487,-1.733192


Refactored function output matches the existing notebook objects.


## 3. Build target vectors  
   - Load survival or outcome labels and construct targets for each split.  
   - Targets must use the same sample IDs and row order as the feature matrices.  
   - Examples:  
       • `y_train`  
       • `y_val`  
       • `y_test`

In [54]:
# Load outcome labels once and index by sample ID for split-specific target construction.

survival_path = Path("../data/raw/TCGA-BRCA.survival.tsv.gz")
event_col = "OS"

surv_df = pd.read_csv(survival_path, sep="\t")[["sample", event_col]].astype({"sample": str}).set_index("sample")

y_train = surv_df.loc[X_clin_train_df.index, event_col].astype(int).copy()
y_val = surv_df.loc[X_clin_val_df.index, event_col].astype(int).copy()
y_test = surv_df.loc[X_clin_test_df.index, event_col].astype(int).copy()

# Validate that target ordering matches the aligned feature matrices exactly.
assert y_train.index.equals(X_clin_train_df.index)
assert y_val.index.equals(X_clin_val_df.index)
assert y_test.index.equals(X_clin_test_df.index)

display(y_train.head(), y_train.tail())
display(y_val.head(), y_val.tail())
display(y_test.head(), y_test.tail())

sample
TCGA-AR-A1AN-01A    0
TCGA-B6-A2IU-01A    0
TCGA-C8-A12Q-01A    1
TCGA-BH-A0E9-01B    0
TCGA-A2-A0ST-01A    0
Name: OS, dtype: int64

sample
TCGA-GM-A2DM-01A    0
TCGA-AR-A2LE-01A    0
TCGA-E2-A1LK-01A    1
TCGA-E9-A1RB-01A    1
TCGA-B6-A0I6-01A    1
Name: OS, dtype: int64

sample
TCGA-BH-A0DD-01A    0
TCGA-A2-A0D3-01A    0
TCGA-OL-A5D6-01A    1
TCGA-A2-A0CO-01A    1
TCGA-A8-A09X-01A    1
Name: OS, dtype: int64

sample
TCGA-AO-A1KR-01A    0
TCGA-AO-A1KP-01A    0
TCGA-B6-A0WZ-01A    0
TCGA-AO-A12B-01A    0
TCGA-AR-A1AH-01A    0
Name: OS, dtype: int64

sample
TCGA-GM-A3XN-01A    0
TCGA-BH-A0DT-01A    0
TCGA-BH-A0B5-01A    0
TCGA-B6-A0WX-01A    1
TCGA-BH-A0C1-01B    1
Name: OS, dtype: int64

sample
TCGA-AR-A2LJ-01A    0
TCGA-BH-A1FU-01A    1
TCGA-BH-A18V-01A    1
TCGA-AR-A2LL-01A    0
TCGA-AR-A5QQ-01A    1
Name: OS, dtype: int64

In [55]:
def build_target_vectors(
    survival_path: str | Path,
    event_col: str,
    X_clin_train_df: pd.DataFrame,
    X_clin_val_df: pd.DataFrame,
    X_clin_test_df: pd.DataFrame,
) -> tuple[pd.Series, pd.Series, pd.Series]:
    """Load outcome labels and construct split-specific target vectors aligned to clinical sample order.

    Parameters
    ----------
    survival_path : str | Path
        Path to the survival table containing `sample` and the outcome column.
    event_col : str
        Name of the outcome column to extract.
    X_clin_train_df, X_clin_val_df, X_clin_test_df : pd.DataFrame
        Clinical feature matrices whose sample indices define the target ordering.

    Returns
    -------
    tuple[pd.Series, pd.Series, pd.Series]
        `y_train`, `y_val`, and `y_test`, indexed by sample ID.
    """
    survival_path = Path(survival_path)

    # Load outcome labels once and index by sample ID for split-specific target construction.
    surv_df = pd.read_csv(survival_path, sep="\t")[["sample", event_col]].astype({"sample": str}).set_index("sample")

    y_train = surv_df.loc[X_clin_train_df.index, event_col].astype(int).copy()
    y_val = surv_df.loc[X_clin_val_df.index, event_col].astype(int).copy()
    y_test = surv_df.loc[X_clin_test_df.index, event_col].astype(int).copy()

    # Validate that target ordering matches the aligned feature matrices exactly.
    assert y_train.index.equals(X_clin_train_df.index)
    assert y_val.index.equals(X_clin_val_df.index)
    assert y_test.index.equals(X_clin_test_df.index)

    display(y_train.head(), y_train.tail())
    display(y_val.head(), y_val.tail())
    display(y_test.head(), y_test.tail())

    return y_train, y_val, y_test


y_train_loaded, y_val_loaded, y_test_loaded = build_target_vectors(
    survival_path=survival_path,
    event_col=event_col,
    X_clin_train_df=X_clin_train_df,
    X_clin_val_df=X_clin_val_df,
    X_clin_test_df=X_clin_test_df,
)

assert y_train_loaded.equals(y_train)
assert y_val_loaded.equals(y_val)
assert y_test_loaded.equals(y_test)

print("Refactored function output matches the existing notebook objects.")


sample
TCGA-AR-A1AN-01A    0
TCGA-B6-A2IU-01A    0
TCGA-C8-A12Q-01A    1
TCGA-BH-A0E9-01B    0
TCGA-A2-A0ST-01A    0
Name: OS, dtype: int64

sample
TCGA-GM-A2DM-01A    0
TCGA-AR-A2LE-01A    0
TCGA-E2-A1LK-01A    1
TCGA-E9-A1RB-01A    1
TCGA-B6-A0I6-01A    1
Name: OS, dtype: int64

sample
TCGA-BH-A0DD-01A    0
TCGA-A2-A0D3-01A    0
TCGA-OL-A5D6-01A    1
TCGA-A2-A0CO-01A    1
TCGA-A8-A09X-01A    1
Name: OS, dtype: int64

sample
TCGA-AO-A1KR-01A    0
TCGA-AO-A1KP-01A    0
TCGA-B6-A0WZ-01A    0
TCGA-AO-A12B-01A    0
TCGA-AR-A1AH-01A    0
Name: OS, dtype: int64

sample
TCGA-GM-A3XN-01A    0
TCGA-BH-A0DT-01A    0
TCGA-BH-A0B5-01A    0
TCGA-B6-A0WX-01A    1
TCGA-BH-A0C1-01B    1
Name: OS, dtype: int64

sample
TCGA-AR-A2LJ-01A    0
TCGA-BH-A1FU-01A    1
TCGA-BH-A18V-01A    1
TCGA-AR-A2LL-01A    0
TCGA-AR-A5QQ-01A    1
Name: OS, dtype: int64

Refactored function output matches the existing notebook objects.


## 4. Validate dataset invariants  
   - Perform sanity checks to confirm that assembled datasets are consistent.  
   - Checks should include:  
       • identical sample counts across modalities and targets  
       • identical sample ordering  
       • no duplicate sample IDs  
       • no missing values introduced during alignment  
       • expected feature counts for each modality

In [56]:
# Check split-by-split consistency across modalities and targets.

for split_name, X_clin_df, X_rna_df, y_split in [
    ("train", X_clin_train_df, X_rna_train_df, y_train),
    ("val", X_clin_val_df, X_rna_val_df, y_val),
    ("test", X_clin_test_df, X_rna_test_df, y_test),
]:
    assert X_clin_df.shape[0] == X_rna_df.shape[0] == y_split.shape[0]
    assert X_clin_df.index.equals(X_rna_df.index)
    assert X_clin_df.index.equals(y_split.index)
    assert not X_clin_df.index.duplicated().any()
    assert not X_rna_df.index.duplicated().any()
    assert not y_split.index.duplicated().any()
    assert X_clin_df.isna().sum().sum() == 0
    assert X_rna_df.isna().sum().sum() == 0
    assert y_split.isna().sum() == 0

# Check that feature dimensions are stable across splits within each modality.
assert X_clin_train_df.shape[1] == X_clin_val_df.shape[1] == X_clin_test_df.shape[1]
assert X_rna_train_df.shape[1] == X_rna_val_df.shape[1] == X_rna_test_df.shape[1]

validation_summary_df = pd.DataFrame(
    {
        "split": ["train", "val", "test"],
        "n_samples": [len(y_train), len(y_val), len(y_test)],
        "n_clin_features": [X_clin_train_df.shape[1], X_clin_val_df.shape[1], X_clin_test_df.shape[1]],
        "n_rna_features": [X_rna_train_df.shape[1], X_rna_val_df.shape[1], X_rna_test_df.shape[1]],
    }
)

display(validation_summary_df.head(), validation_summary_df.tail())
print("All dataset invariant checks passed.")

,split,n_samples,n_clin_features,n_rna_features
0,train,203,98,25431
1,val,43,98,25431
2,test,44,98,25431


,split,n_samples,n_clin_features,n_rna_features
0,train,203,98,25431
1,val,43,98,25431
2,test,44,98,25431


All dataset invariant checks passed.


In [57]:
def validate_dataset_invariants(
    X_clin_train_df: pd.DataFrame,
    X_clin_val_df: pd.DataFrame,
    X_clin_test_df: pd.DataFrame,
    X_rna_train_df: pd.DataFrame,
    X_rna_val_df: pd.DataFrame,
    X_rna_test_df: pd.DataFrame,
    y_train: pd.Series,
    y_val: pd.Series,
    y_test: pd.Series,
) -> pd.DataFrame:
    """Validate split-level consistency across modalities and target vectors.

    Parameters
    ----------
    X_clin_train_df, X_clin_val_df, X_clin_test_df : pd.DataFrame
        Clinical feature matrices for the train, validation, and test splits.
    X_rna_train_df, X_rna_val_df, X_rna_test_df : pd.DataFrame
        RNA feature matrices for the train, validation, and test splits.
    y_train, y_val, y_test : pd.Series
        Target vectors aligned to the split-specific feature matrices.

    Returns
    -------
    pd.DataFrame
        Split-level summary of sample counts and feature counts after validation.
    """
    # Check split-by-split consistency across modalities and targets.
    for split_name, X_clin_df, X_rna_df, y_split in [
        ("train", X_clin_train_df, X_rna_train_df, y_train),
        ("val", X_clin_val_df, X_rna_val_df, y_val),
        ("test", X_clin_test_df, X_rna_test_df, y_test),
    ]:
        assert X_clin_df.shape[0] == X_rna_df.shape[0] == y_split.shape[0]
        assert X_clin_df.index.equals(X_rna_df.index)
        assert X_clin_df.index.equals(y_split.index)
        assert not X_clin_df.index.duplicated().any()
        assert not X_rna_df.index.duplicated().any()
        assert not y_split.index.duplicated().any()
        assert X_clin_df.isna().sum().sum() == 0
        assert X_rna_df.isna().sum().sum() == 0
        assert y_split.isna().sum() == 0

    # Check that feature dimensions are stable across splits within each modality.
    assert X_clin_train_df.shape[1] == X_clin_val_df.shape[1] == X_clin_test_df.shape[1]
    assert X_rna_train_df.shape[1] == X_rna_val_df.shape[1] == X_rna_test_df.shape[1]

    validation_summary_df = pd.DataFrame(
        {
            "split": ["train", "val", "test"],
            "n_samples": [len(y_train), len(y_val), len(y_test)],
            "n_clin_features": [X_clin_train_df.shape[1], X_clin_val_df.shape[1], X_clin_test_df.shape[1]],
            "n_rna_features": [X_rna_train_df.shape[1], X_rna_val_df.shape[1], X_rna_test_df.shape[1]],
        }
    )

    display(validation_summary_df.head(), validation_summary_df.tail())
    print("All dataset invariant checks passed.")

    return validation_summary_df


validation_summary_loaded_df = validate_dataset_invariants(
    X_clin_train_df=X_clin_train_df,
    X_clin_val_df=X_clin_val_df,
    X_clin_test_df=X_clin_test_df,
    X_rna_train_df=X_rna_train_df,
    X_rna_val_df=X_rna_val_df,
    X_rna_test_df=X_rna_test_df,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
)

assert validation_summary_loaded_df.equals(validation_summary_df)

print("Refactored function output matches the existing notebook objects.")

,split,n_samples,n_clin_features,n_rna_features
0,train,203,98,25431
1,val,43,98,25431
2,test,44,98,25431


,split,n_samples,n_clin_features,n_rna_features
0,train,203,98,25431
1,val,43,98,25431
2,test,44,98,25431


All dataset invariant checks passed.
Refactored function output matches the existing notebook objects.



## 5. Save final model inputs  
   - Write the assembled datasets for downstream modeling.  
   - Save modality-specific inputs for each split.  
   - Optionally also save concatenated feature matrices for baseline models.

In [58]:
# Save split-specific modality inputs and targets for downstream modeling.

outdir = Path("../data/tests/assembled_notebook_test")
for split_name in ["train", "val", "test"]:
    (outdir / split_name).mkdir(parents=True, exist_ok=True)

X_clin_train_df.to_parquet(outdir / "train" / "X_clinical.parquet")
X_clin_val_df.to_parquet(outdir / "val" / "X_clinical.parquet")
X_clin_test_df.to_parquet(outdir / "test" / "X_clinical.parquet")

X_rna_train_df.to_parquet(outdir / "train" / "X_rna.parquet")
X_rna_val_df.to_parquet(outdir / "val" / "X_rna.parquet")
X_rna_test_df.to_parquet(outdir / "test" / "X_rna.parquet")

y_train.to_frame(name="y").to_parquet(outdir / "train" / "y.parquet")
y_val.to_frame(name="y").to_parquet(outdir / "val" / "y.parquet")
y_test.to_frame(name="y").to_parquet(outdir / "test" / "y.parquet")

In [59]:
def save_final_model_inputs(
    outdir: str | Path,
    X_clin_train_df: pd.DataFrame,
    X_clin_val_df: pd.DataFrame,
    X_clin_test_df: pd.DataFrame,
    X_rna_train_df: pd.DataFrame,
    X_rna_val_df: pd.DataFrame,
    X_rna_test_df: pd.DataFrame,
    y_train: pd.Series,
    y_val: pd.Series,
    y_test: pd.Series,
) -> tuple[Path, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Save split-specific modality inputs, targets, and concatenated baseline features.

    Parameters
    ----------
    outdir : str | Path
        Base output directory for assembled split-specific parquet files.
    X_clin_train_df, X_clin_val_df, X_clin_test_df : pd.DataFrame
        Clinical feature matrices for the train, validation, and test splits.
    X_rna_train_df, X_rna_val_df, X_rna_test_df : pd.DataFrame
        RNA feature matrices for the train, validation, and test splits.
    y_train, y_val, y_test : pd.Series
        Target vectors for the train, validation, and test splits.

    Returns
    -------
    tuple[Path, pd.DataFrame, pd.DataFrame, pd.DataFrame]
        Output directory plus concatenated train, validation, and test feature matrices.
    """
    outdir = Path(outdir)

    # Save split-specific modality inputs and targets for downstream modeling.
    for split_name in ["train", "val", "test"]:
        (outdir / split_name).mkdir(parents=True, exist_ok=True)

    X_clin_train_df.to_parquet(outdir / "train" / "X_clinical.parquet")
    X_clin_val_df.to_parquet(outdir / "val" / "X_clinical.parquet")
    X_clin_test_df.to_parquet(outdir / "test" / "X_clinical.parquet")

    X_rna_train_df.to_parquet(outdir / "train" / "X_rna.parquet")
    X_rna_val_df.to_parquet(outdir / "val" / "X_rna.parquet")
    X_rna_test_df.to_parquet(outdir / "test" / "X_rna.parquet")

    y_train.to_frame(name="y").to_parquet(outdir / "train" / "y.parquet")
    y_val.to_frame(name="y").to_parquet(outdir / "val" / "y.parquet")
    y_test.to_frame(name="y").to_parquet(outdir / "test" / "y.parquet")

    # Optionally save concatenated feature matrices for unimodal-vs-fusion baselines.
    X_concat_train_df = pd.concat([X_clin_train_df, X_rna_train_df], axis=1)
    X_concat_val_df = pd.concat([X_clin_val_df, X_rna_val_df], axis=1)
    X_concat_test_df = pd.concat([X_clin_test_df, X_rna_test_df], axis=1)

    X_concat_train_df.to_parquet(outdir / "train" / "X_concat.parquet")
    X_concat_val_df.to_parquet(outdir / "val" / "X_concat.parquet")
    X_concat_test_df.to_parquet(outdir / "test" / "X_concat.parquet")

    display(X_concat_train_df.head(), X_concat_train_df.tail())
    print(f"Saved assembled datasets to: {outdir}")

    return outdir, X_concat_train_df, X_concat_val_df, X_concat_test_df


assembled_test_outdir = Path("../data/tests/assembled_function_test")

(
    assembled_test_outdir,
    X_concat_train_loaded_df,
    X_concat_val_loaded_df,
    X_concat_test_loaded_df,
) = save_final_model_inputs(
    outdir=assembled_test_outdir,
    X_clin_train_df=X_clin_train_df,
    X_clin_val_df=X_clin_val_df,
    X_clin_test_df=X_clin_test_df,
    X_rna_train_df=X_rna_train_df,
    X_rna_val_df=X_rna_val_df,
    X_rna_test_df=X_rna_test_df,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
)

assert pd.read_parquet(assembled_test_outdir / "train" / "X_clinical.parquet").equals(X_clin_train_df)
assert pd.read_parquet(assembled_test_outdir / "val" / "X_clinical.parquet").equals(X_clin_val_df)
assert pd.read_parquet(assembled_test_outdir / "test" / "X_clinical.parquet").equals(X_clin_test_df)

assert pd.read_parquet(assembled_test_outdir / "train" / "X_rna.parquet").equals(X_rna_train_df)
assert pd.read_parquet(assembled_test_outdir / "val" / "X_rna.parquet").equals(X_rna_val_df)
assert pd.read_parquet(assembled_test_outdir / "test" / "X_rna.parquet").equals(X_rna_test_df)

assert pd.read_parquet(assembled_test_outdir / "train" / "y.parquet")["y"].equals(y_train.rename("y"))
assert pd.read_parquet(assembled_test_outdir / "val" / "y.parquet")["y"].equals(y_val.rename("y"))
assert pd.read_parquet(assembled_test_outdir / "test" / "y.parquet")["y"].equals(y_test.rename("y"))

assert pd.read_parquet(assembled_test_outdir / "train" / "X_concat.parquet").equals(X_concat_train_loaded_df)
assert pd.read_parquet(assembled_test_outdir / "val" / "X_concat.parquet").equals(X_concat_val_loaded_df)
assert pd.read_parquet(assembled_test_outdir / "test" / "X_concat.parquet").equals(X_concat_test_loaded_df)

print("Refactored function output matches the written parquet files.")


,year_of_birth.demographic,year_of_diagnosis.diagnoses,age_at_earliest_diagnosis_in_years.diagnoses.xena_derived,days_to_collection.samples,initial_weight.samples,oct_embedded.samples,ajcc_pathologic_m.diagnoses_M0,ajcc_pathologic_m.diagnoses_M1,ajcc_pathologic_m.diagnoses_MX,ajcc_pathologic_m.diagnoses_cM0 (i+),...,ENSG00000288573.1,ENSG00000288585.1,ENSG00000288586.1,ENSG00000288596.2,ENSG00000288600.1,ENSG00000288605.1,ENSG00000288612.1,ENSG00000288658.1,ENSG00000288670.1,ENSG00000288675.1
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-AR-A1AN-01A,1960.0,2006.0,46.504110,1463.0,70.0,1,1.0,0.0,0.0,0.0,...,-1.353641,-0.352161,0.247873,-1.847424,-0.651507,-0.606533,-0.440411,0.176202,0.426529,1.225862
TCGA-B6-A2IU-01A,1936.0,1998.0,62.753425,4665.0,140.0,1,1.0,0.0,0.0,0.0,...,0.012429,1.103978,1.517371,0.609031,0.524253,-0.474512,-0.202529,-0.660575,0.394027,-0.961233
TCGA-C8-A12Q-01A,1932.0,2010.0,78.720548,57.0,810.0,1,1.0,0.0,0.0,0.0,...,-1.509343,-0.020435,-0.527875,-0.585608,0.204504,-0.378965,2.682691,-0.551112,0.489490,-1.008443
TCGA-BH-A0E9-01B,1953.0,2006.0,53.191781,1357.0,480.0,1,1.0,0.0,0.0,0.0,...,0.585732,2.531337,-0.092806,0.688908,-0.607627,-0.034685,1.212378,-0.406525,0.965263,0.145407
TCGA-A2-A0ST-01A,1941.0,2003.0,62.800000,2597.0,210.0,1,1.0,0.0,0.0,0.0,...,0.178092,-0.914768,-1.051409,-0.765611,2.093778,-0.025683,-0.928029,0.542307,-1.556974,-0.272568


,year_of_birth.demographic,year_of_diagnosis.diagnoses,age_at_earliest_diagnosis_in_years.diagnoses.xena_derived,days_to_collection.samples,initial_weight.samples,oct_embedded.samples,ajcc_pathologic_m.diagnoses_M0,ajcc_pathologic_m.diagnoses_M1,ajcc_pathologic_m.diagnoses_MX,ajcc_pathologic_m.diagnoses_cM0 (i+),...,ENSG00000288573.1,ENSG00000288585.1,ENSG00000288586.1,ENSG00000288596.2,ENSG00000288600.1,ENSG00000288605.1,ENSG00000288612.1,ENSG00000288658.1,ENSG00000288670.1,ENSG00000288675.1
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-GM-A2DM-01A,1947.0,2004.0,57.972603,2464.0,120.0,0,1.0,0.0,0.0,0.0,...,0.265048,-0.884606,0.141415,-0.480243,-0.579166,-0.528830,-0.286348,-0.517218,0.045952,0.367215
TCGA-AR-A2LE-01A,1932.0,2001.0,69.860274,3755.0,320.0,1,1.0,0.0,0.0,0.0,...,0.236000,-0.231015,0.883146,-0.100171,-0.352117,-0.606533,-0.897766,3.383043,0.066095,-0.709205
TCGA-E2-A1LK-01A,1924.0,2008.0,84.265753,1051.0,70.0,0,1.0,0.0,0.0,0.0,...,0.110951,-0.607541,0.610262,1.347791,-0.521076,1.184644,1.959450,1.703494,0.349865,-0.400755
TCGA-E9-A1RB-01A,1971.0,2011.0,40.424658,39.0,490.0,1,1.0,0.0,0.0,0.0,...,-1.007468,-0.571601,-1.173723,0.221939,-0.343172,-0.560550,-0.506072,-0.084724,0.648266,-0.360840
TCGA-B6-A0I6-01A,1941.0,1990.0,49.353425,7362.0,200.0,1,1.0,0.0,0.0,0.0,...,0.366692,-0.945813,2.813252,0.383612,-0.208193,0.491286,-1.132417,1.301828,-0.865487,-1.733192


Saved assembled datasets to: ../data/tests/assembled_function_test
Refactored function output matches the written parquet files.


## 6. Test the assemble dataset module

In [60]:
# Test the assemble dataset module

# Run the assembly module through its CLI entry point.
assembled_outdir = Path("../data/processed/assembled")
subprocess.run(
    [
        sys.executable,
        "../scripts/assemble_dataset.py",
        "--clinical-dir",
        "../data/processed/clinical",
        "--rna-dir",
        "../data/processed/rna",
        "--survival-path",
        "../data/raw/TCGA-BRCA.survival.tsv.gz",
        "--event-col",
        "OS",
        "--outdir",
        str(assembled_outdir),
    ],
    check=True,
)

# Load saved outputs and metadata for a minimal end-to-end verification.
X_clin_train_saved_df = pd.read_parquet(assembled_outdir / "train" / "X_clinical.parquet")
X_clin_val_saved_df = pd.read_parquet(assembled_outdir / "val" / "X_clinical.parquet")
X_clin_test_saved_df = pd.read_parquet(assembled_outdir / "test" / "X_clinical.parquet")

X_rna_train_saved_df = pd.read_parquet(assembled_outdir / "train" / "X_rna.parquet")
X_rna_val_saved_df = pd.read_parquet(assembled_outdir / "val" / "X_rna.parquet")
X_rna_test_saved_df = pd.read_parquet(assembled_outdir / "test" / "X_rna.parquet")

X_concat_train_saved_df = pd.read_parquet(assembled_outdir / "train" / "X_concat.parquet")
X_concat_val_saved_df = pd.read_parquet(assembled_outdir / "val" / "X_concat.parquet")
X_concat_test_saved_df = pd.read_parquet(assembled_outdir / "test" / "X_concat.parquet")

y_train_saved = pd.read_parquet(assembled_outdir / "train" / "y.parquet")["y"]
y_val_saved = pd.read_parquet(assembled_outdir / "val" / "y.parquet")["y"]
y_test_saved = pd.read_parquet(assembled_outdir / "test" / "y.parquet")["y"]

metadata = json.loads((assembled_outdir / "assemble_dataset_metadata.json").read_text(encoding="utf-8"))

# Check shapes, ordering, and feature layout against the notebook objects.
assert X_clin_train_saved_df.shape == X_clin_train_df.shape
assert X_clin_val_saved_df.shape == X_clin_val_df.shape
assert X_clin_test_saved_df.shape == X_clin_test_df.shape

assert X_rna_train_saved_df.shape == X_rna_train_df.shape
assert X_rna_val_saved_df.shape == X_rna_val_df.shape
assert X_rna_test_saved_df.shape == X_rna_test_df.shape

assert X_clin_train_saved_df.columns.equals(X_clin_train_df.columns)
assert X_clin_val_saved_df.columns.equals(X_clin_val_df.columns)
assert X_clin_test_saved_df.columns.equals(X_clin_test_df.columns)

assert X_rna_train_saved_df.columns.equals(X_rna_train_df.columns)
assert X_rna_val_saved_df.columns.equals(X_rna_val_df.columns)
assert X_rna_test_saved_df.columns.equals(X_rna_test_df.columns)

assert X_clin_train_saved_df.index.equals(X_clin_train_df.index)
assert X_clin_val_saved_df.index.equals(X_clin_val_df.index)
assert X_clin_test_saved_df.index.equals(X_clin_test_df.index)

assert X_rna_train_saved_df.index.equals(X_rna_train_df.index)
assert X_rna_val_saved_df.index.equals(X_rna_val_df.index)
assert X_rna_test_saved_df.index.equals(X_rna_test_df.index)

assert y_train_saved.equals(y_train.rename("y"))
assert y_val_saved.equals(y_val.rename("y"))
assert y_test_saved.equals(y_test.rename("y"))

assert X_concat_train_saved_df.shape[1] == X_clin_train_df.shape[1] + X_rna_train_df.shape[1]
assert X_concat_val_saved_df.shape[1] == X_clin_val_df.shape[1] + X_rna_val_df.shape[1]
assert X_concat_test_saved_df.shape[1] == X_clin_test_df.shape[1] + X_rna_test_df.shape[1]

# Verify key metadata fields and recorded dataset statistics.
assert metadata["script_name"] == "assemble_dataset.py"
assert metadata["key_parameters_used"]["event_col"] == event_col
assert Path(metadata["output_files"]["train_clinical_path"]).name == "X_clinical.parquet"
assert metadata["dataset_statistics"]["n_samples_train"] == X_clin_train_df.shape[0]
assert metadata["dataset_statistics"]["n_samples_val"] == X_clin_val_df.shape[0]
assert metadata["dataset_statistics"]["n_samples_test"] == X_clin_test_df.shape[0]
assert metadata["dataset_statistics"]["n_clin_features_train"] == X_clin_train_df.shape[1]
assert metadata["dataset_statistics"]["n_rna_features_train"] == X_rna_train_df.shape[1]
assert metadata["dataset_statistics"]["n_concat_features_train"] == X_concat_train_saved_df.shape[1]
assert len(metadata["validation_summary"]) == 3

print("Assemble dataset smoke test passed.")


clinical: train=(203, 98), val=(43, 98), test=(44, 98)
rna: train=(203, 25431), val=(43, 25431), test=(44, 25431)
All dataset invariant checks passed.
Saved assembled datasets to: ../data/processed/assembled
Assemble dataset smoke test passed.
